# ChildSafe — detector bút và đũa
Upload childsafe_pen_chopsticks_detection.zip thành Kaggle Dataset, thêm vào notebook.
Bật GPU và Internet. Dữ liệu là ảnh nguyên khung cùng bounding box, không phải crop phân loại.
Train/val/test giữ split chính thức của Open Images. Test chỉ dùng đánh giá cuối.
Đây là detector hộp: cần mask hoặc nhãn đầu vật bổ sung trước khi xác nhận khoảng cách đầu vật–miệng.


In [ ]:
%pip install -q ultralytics==8.3.253


In [ ]:
from pathlib import Path
import os, shutil, yaml, torch
assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook settings'
matches=list(Path('/kaggle/input').rglob('manifest.csv'))
matches=[p.parent for p in matches if (p.parent/'report.json').is_file() and (p.parent/'images/train').is_dir()]
assert len(matches)==1, f'Expected one detector dataset, found {matches}'
source=matches[0]
work=Path('/kaggle/working/childsafe_detector'); work.mkdir(exist_ok=True)
os.chdir(work)
(work/'models').mkdir(exist_ok=True)
weights=list(Path('/kaggle/input').rglob('yolo11n-seg.pt'))
assert len(weights)==1, 'Dataset ZIP must contain models/yolo11n-seg.pt'
shutil.copy2(weights[0],work/'models/yolo11n-seg.pt')
cfg={'path':str(source),'train':'images/train','val':'images/val','test':'images/test','names':{0:'pen',1:'chopsticks'}}
Path('dataset.yaml').write_text(yaml.safe_dump(cfg),encoding='utf-8')
print((source/'report.json').read_text())


In [ ]:
"""Train a separate small-object detector; boxes are not segmentation masks."""
import argparse
import json
from pathlib import Path


def train(data, output, epochs, size, batch, device):
    if output.exists():
        raise FileExistsError('training output exists; choose a new --output directory')
    from ultralytics import YOLO
    # Transfer the existing COCO backbone into a detection architecture. The
    # two-class detection head is trained afresh; no weight download is needed.
    model=YOLO('yolo11n.yaml').load('models/yolo11n-seg.pt')
    model.train(data=str(data),epochs=epochs,imgsz=size,batch=batch,device=device,
                workers=0,project=str(output.parent),name=output.name,exist_ok=False,
                seed=42,deterministic=True,patience=10,amp=False,plots=False,
                cache=False,close_mosaic=3,save=True)
    best=Path(model.trainer.best)
    evaluated=YOLO(str(best))
    result=evaluated.val(data=str(data),split='test',device=device,imgsz=size,plots=False,
                         project=str(output),name='heldout_test')
    report={'checkpoint':str(best.resolve()),'classes':evaluated.names,'epochs_requested':epochs,
            'image_size':size,'test_metrics':result.results_dict,
            'limitation':'Bounding-box baseline. Does not localize object tips or confirm mask proximity.'}
    report['per_class']={evaluated.names[int(class_id)]:dict(precision=float(result.box.p[i]),
                        recall=float(result.box.r[i]),map50=float(result.box.ap50[i]),
                        map50_95=float(result.box.ap[i]))
                        for i,class_id in enumerate(result.box.ap_class_index)}
    output.mkdir(parents=True,exist_ok=True)
    (output/'evaluation.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    print(json.dumps(report,indent=2),flush=True)




In [ ]:
train(Path('dataset.yaml'), Path('outputs/pen_chopsticks_detector_gpu'), epochs=50, size=640, batch=16, device='0')


In [ ]:
import zipfile
with zipfile.ZipFile('/kaggle/working/pen_chopsticks_detector_results.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for p in Path('outputs').rglob('*'):
        if p.is_file(): archive.write(p,p.as_posix())
print('Download pen_chopsticks_detector_results.zip from Kaggle Output')
